# Master Brain (Work Computer) – API Bootstrap

This notebook is tailored to your work computer paths:

- **Master Brain root**: `C:/Users/dunnmk/Dropbox (Personal)/Apps/Master Brain`
- **API repo**: `C:/Users/dunnmk/Dropbox (Personal)/Apps/Master-Brain-API (Selective Sync Conflict 2)`

The goal is to (1) reuse the existing local index if it already exists, (2) start the Bridge API, and (3) confirm it’s reachable via `/health` in a couple seconds.

If you previously built the index on this device, you should *not* need to re-upload anything—this just reuses the `.pkl` file on disk.

In [1]:
from __future__ import annotations

import os
import sys
from pathlib import Path

MASTER_BRAIN_ROOT = Path(r"C:\Users\dunnmk\Dropbox (Personal)\Apps\Master Brain")
API_REPO_ROOT = Path(r"C:\Users\dunnmk\Dropbox (Personal)\Apps\Master-Brain-API (Selective Sync Conflict 2)")

BRIDGE_HOST = os.environ.get("BRIDGE_HOST", "127.0.0.1").strip() or "127.0.0.1"
BRIDGE_PORT = int((os.environ.get("BRIDGE_PORT", "8787") or "8787").strip())

# The API defaults to this key if BRIDGE_API_KEY is missing/placeholder.
BRIDGE_API_KEY = (os.environ.get("BRIDGE_API_KEY") or "master-brain-bridge-local").strip()

# Default index is normally <repo>/data/master_brain_index.pkl (unless you override BRIDGE_DEFAULT_INDEX_PATH).
DEFAULT_INDEX = API_REPO_ROOT / "data" / "master_brain_index.pkl"
INDEX_PATH = Path(os.environ.get("BRIDGE_DEFAULT_INDEX_PATH", str(DEFAULT_INDEX))).expanduser()

print("Python:         " + sys.executable)
print("API repo root:   " + str(API_REPO_ROOT))
print("Master Brain:    " + str(MASTER_BRAIN_ROOT))
print("Index path:      " + str(INDEX_PATH))
print("Bridge endpoint: http://%s:%s" % (BRIDGE_HOST, BRIDGE_PORT))
print("API key:         " + (BRIDGE_API_KEY[:4] + "…" if BRIDGE_API_KEY else "<none>"))

if not API_REPO_ROOT.exists():
    raise FileNotFoundError(f"API repo not found at: {API_REPO_ROOT}")
if not MASTER_BRAIN_ROOT.exists():
    print(f"WARNING: Master Brain root not found at: {MASTER_BRAIN_ROOT}")

Python:         c:\Users\dunnmk\miniforge3\envs\wilsontew-jupyter-kernels\python.exe
API repo root:   C:\Users\dunnmk\Dropbox (Personal)\Apps\Master-Brain-API (Selective Sync Conflict 2)
Master Brain:    C:\Users\dunnmk\Dropbox (Personal)\Apps\Master Brain
Index path:      C:\Users\dunnmk\Dropbox (Personal)\Apps\Master-Brain-API (Selective Sync Conflict 2)\data\master_brain_index.pkl
Bridge endpoint: http://127.0.0.1:8787
API key:         mast…


In [2]:
# Check whether the index already exists (if it does, you do NOT need to rebuild).

if INDEX_PATH.exists():
    print(f"OK: index exists ({INDEX_PATH.stat().st_size / 1e6:.1f} MB)")
else:
    print("Index missing at: " + str(INDEX_PATH))
    print("If this machine already ran the build, double-check BRIDGE_DEFAULT_INDEX_PATH and that Selective Sync kept the data/ folder.")
    print("If you DO need to rebuild, run a build from the API repo root (example):")
    print("  python -m math_logic_agent.cli build-master-brain --master-root \"%s\" --module-config config/master_brain.toml --index-path data/master_brain_index.pkl --incremental" % MASTER_BRAIN_ROOT)

Index missing at: C:\Users\dunnmk\Dropbox (Personal)\Apps\Master-Brain-API (Selective Sync Conflict 2)\data\master_brain_index.pkl
If this machine already ran the build, double-check BRIDGE_DEFAULT_INDEX_PATH and that Selective Sync kept the data/ folder.
If you DO need to rebuild, run a build from the API repo root (example):
  python -m math_logic_agent.cli build-master-brain --master-root "C:\Users\dunnmk\Dropbox (Personal)\Apps\Master Brain" --module-config config/master_brain.toml --index-path data/master_brain_index.pkl --incremental


## Start the Bridge API

This cell starts `uvicorn math_logic_agent.api:app` using the repo’s virtualenv if present (otherwise it uses the current Python).

On Windows, it tries to open the API in a **new console window** so it can keep running while you continue using this notebook.

If port `8787` is already taken, change `BRIDGE_PORT` (or set `BRIDGE_PORT` in your environment) and rerun Cell 2.

In [3]:
import subprocess

venv_python = API_REPO_ROOT / ".venv" / "Scripts" / "python.exe"
python_exe = str(venv_python) if venv_python.exists() else sys.executable

env = dict(os.environ)
env["BRIDGE_HOST"] = BRIDGE_HOST
env["BRIDGE_PORT"] = str(BRIDGE_PORT)
env["BRIDGE_API_KEY"] = BRIDGE_API_KEY
env["BRIDGE_WORKSPACE_ROOT"] = str(API_REPO_ROOT)
env["BRIDGE_DEFAULT_INDEX_PATH"] = str(INDEX_PATH)

cmd = [
    python_exe,
    "-m",
    "uvicorn",
    "math_logic_agent.api:app",
    "--host",
    BRIDGE_HOST,
    "--port",
    str(BRIDGE_PORT),
]

print("Starting API with:")
print("  cwd: " + str(API_REPO_ROOT))
print("  py:  " + python_exe)
print("  cmd: " + " ".join(cmd))

# Try to start in a new console so it keeps running independently of the notebook kernel.
creationflags = 0
try:
    creationflags = subprocess.CREATE_NEW_CONSOLE  # type: ignore[attr-defined]
except Exception:
    creationflags = 0

proc = subprocess.Popen(cmd, cwd=str(API_REPO_ROOT), env=env, creationflags=creationflags)
print(f"Spawned uvicorn PID: {proc.pid}")
print(f"Now check: http://{BRIDGE_HOST}:{BRIDGE_PORT}/health")

Starting API with:
  cwd: C:\Users\dunnmk\Dropbox (Personal)\Apps\Master-Brain-API (Selective Sync Conflict 2)
  py:  C:\Users\dunnmk\Dropbox (Personal)\Apps\Master-Brain-API (Selective Sync Conflict 2)\.venv\Scripts\python.exe
  cmd: C:\Users\dunnmk\Dropbox (Personal)\Apps\Master-Brain-API (Selective Sync Conflict 2)\.venv\Scripts\python.exe -m uvicorn math_logic_agent.api:app --host 127.0.0.1 --port 8787
Spawned uvicorn PID: 21804
Now check: http://127.0.0.1:8787/health


## Verify reachability (`/health`)

If the API is running, this should return JSON with `status: ok`.

In [4]:
import json
from urllib import request

url = f"http://{BRIDGE_HOST}:{BRIDGE_PORT}/health"

try:
    with request.urlopen(url, timeout=3) as resp:
        payload = resp.read().decode("utf-8", errors="replace")
    data = json.loads(payload)
    print(json.dumps(data, indent=2))
except Exception as e:
    print(f"ERROR calling {url}: {type(e).__name__}: {e}")
    print("If you just started uvicorn, wait 1–2 seconds and run this cell again.")

ERROR calling http://127.0.0.1:8787/health: URLError: <urlopen error [WinError 10061] No connection could be made because the target machine actively refused it>
If you just started uvicorn, wait 1–2 seconds and run this cell again.


## Smoke test a query

This hits `POST /v1/query` using the API key header. If you didn’t set a custom key, `master-brain-bridge-local` should work.

In [5]:
import json
from urllib import request

url = f"http://{BRIDGE_HOST}:{BRIDGE_PORT}/v1/query"
body = {"question": "What is SVD used for in denoising?", "k": 6}

req = request.Request(
    url=url,
    data=json.dumps(body).encode("utf-8"),
    method="POST",
    headers={
        "Content-Type": "application/json",
        "x-api-key": BRIDGE_API_KEY,
    },
)

try:
    with request.urlopen(req, timeout=15) as resp:
        payload = resp.read().decode("utf-8", errors="replace")
    data = json.loads(payload)
    print("mode:", data.get("mode"))
    print("confidence:", data.get("confidence"), data.get("confidence_label"))
    print("modules:", data.get("selected_modules"))
    print()
    print(data.get("answer", "")[:1200])
except Exception as e:
    print(f"ERROR calling {url}: {type(e).__name__}: {e}")
    print("Common causes: index missing/unreadable, API key mismatch, or the API isn’t running.")

ERROR calling http://127.0.0.1:8787/v1/query: URLError: <urlopen error [WinError 10061] No connection could be made because the target machine actively refused it>
Common causes: index missing/unreadable, API key mismatch, or the API isn’t running.
